In [1]:
!pip install -q langchain rouge-score pandas scikit-learn

In [2]:
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate
from rouge_score import rouge_scorer
from sklearn.metrics import accuracy_score
import pandas as pd
import json

TASK 1: ZERO-SHOT SUMMARIZATION

In [3]:
# Zero-Shot Prompt

zero_shot_prompt = PromptTemplate(
    input_variables=["text"],
    template="""
You are a Financial Analyst.

Summarize the following earnings call in 3 bullet points.

{text}
"""
)

In [4]:
# Mock Summarizer Function

def mock_summarizer(text):
    sentences = text.split(".")

    summary = []

    for s in sentences[:3]:
        s = s.strip()

        if s:
            summary.append(f"- {s}")

    return "\n".join(summary)

In [15]:
earnings_call = """
Revenue increased by 20%.
Cloud business expanded rapidly.
Operating profit improved.
Customer growth remained strong.
"""

formatted_prompt = zero_shot_prompt.format(
    text=earnings_call
)

print("\n","PROMPT:\n")
print(formatted_prompt)

print("\nSUMMARY:\n")
print(mock_summarizer(earnings_call))


 PROMPT:


You are a Financial Analyst.

Summarize the following earnings call in 3 bullet points.


Revenue increased by 20%.
Cloud business expanded rapidly.
Operating profit improved.
Customer growth remained strong.



SUMMARY:

- Revenue increased by 20%
- Cloud business expanded rapidly
- Operating profit improved


TASK 1: FEW-SHOT SUMMARIZATION

In [16]:
examples = [

{
"text":"Revenue grew 15%. Cloud adoption increased.",
"summary":"- Revenue up\n- Cloud growth\n- Strong performance"
},

{
"text":"Expenses increased. Margins declined.",
"summary":"- Costs increased\n- Margins fell\n- Profitability impacted"
}
]

In [17]:
example_prompt = PromptTemplate(
    input_variables=["text","summary"],
    template="""
Text:
{text}

Summary:
{summary}
"""
)

few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,

    prefix="Learn from examples.",

    suffix="""
Text:
{text}

Summary:
""",

    input_variables=["text"]
)

In [18]:
test_text = """
AI products launched successfully.
Enterprise demand increased.
Revenue exceeded forecasts.
"""

print(
    few_shot_prompt.format(
        text=test_text
    )
)

print("\nGenerated Summary:\n")
print(mock_summarizer(test_text))

Learn from examples.


Text:
Revenue grew 15%. Cloud adoption increased.

Summary:
- Revenue up
- Cloud growth
- Strong performance



Text:
Expenses increased. Margins declined.

Summary:
- Costs increased
- Margins fell
- Profitability impacted



Text:

AI products launched successfully.
Enterprise demand increased.
Revenue exceeded forecasts.


Summary:


Generated Summary:

- AI products launched successfully
- Enterprise demand increased
- Revenue exceeded forecasts


TASK 3: TICKET CLASSIFIER

In [19]:
classifier_prompt = PromptTemplate(
    input_variables=["ticket"],
    template="""
Classify the support ticket into:

Billing
Tech
Refund
General
Escalate

Ticket:
{ticket}
"""
)

In [20]:
def classify_ticket(ticket):

    ticket = ticket.lower()

    if "refund" in ticket:
        return "Refund"

    elif "charged" in ticket or "invoice" in ticket:
        return "Billing"

    elif "login" in ticket or "password" in ticket:
        return "Tech"

    elif "legal" in ticket:
        return "Escalate"

    elif "security" in ticket:
        return "Escalate"

    else:
        return "General"

In [21]:
tickets = [

"I was charged twice.",

"Unable to login.",

"Need refund immediately.",

"What are support timings?",

"Legal complaint submitted."
]

for ticket in tickets:

    print("\nTicket:", ticket)

    print(
        "Category:",
        classify_ticket(ticket)
    )


Ticket: I was charged twice.
Category: Billing

Ticket: Unable to login.
Category: Tech

Ticket: Need refund immediately.
Category: Refund

Ticket: What are support timings?
Category: General

Ticket: Legal complaint submitted.
Category: Escalate


TASK 4: ROUGE-L EVALUATION

In [22]:
reference = """
Revenue increased due to strong cloud growth.
"""

generated = """
Revenue increased because cloud services grew.
"""

scorer = rouge_scorer.RougeScorer(
    ['rougeL'],
    use_stemmer=True
)

score = scorer.score(
    reference,
    generated
)

print(score["rougeL"])

Score(precision=0.5, recall=0.42857142857142855, fmeasure=0.4615384615384615)


In [23]:
#PROMPT LOGGING
prompt_logs = []

prompt_logs.append(
    {
        "task":"summarization",
        "prompt":formatted_prompt
    }
)

print("\n",prompt_logs)




 [{'task': 'summarization', 'prompt': '\nYou are a Financial Analyst.\n\nSummarize the following earnings call in 3 bullet points.\n\n\nRevenue increased by 20%.\nCloud business expanded rapidly.\nOperating profit improved.\nCustomer growth remained strong.\n\n'}]


EXTENSION TASK

In [26]:
#Along with category prediction, assign a confidence score between 1–5.

def classify_with_confidence(ticket):

    ticket = ticket.lower()

    if "refund" in ticket:
        return {
            "category":"Refund",
            "confidence":5
        }

    elif "charged" in ticket or "invoice" in ticket:
        return {
            "category":"Billing",
            "confidence":5
        }

    elif "login" in ticket or "password" in ticket:
        return {
            "category":"Tech",
            "confidence":4
        }

    elif "legal" in ticket or "security" in ticket:
        return {
            "category":"Escalate",
            "confidence":5
        }

    else:
        return {
            "category":"General",
            "confidence":3
        }


ticket = "Unable to login after password reset"

print(classify_with_confidence(ticket))

{'category': 'Tech', 'confidence': 4}


In [27]:
#Recommend resolution time based on category.
def get_sla(category):

    sla_rules = {

        "Billing":24,
        "Tech":12,
        "Refund":48,
        "General":72,
        "Escalate":4
    }

    return sla_rules[category]

In [28]:
#test

result = classify_with_confidence(
    "I need a refund immediately"
)

sla = get_sla(result["category"])

print("Category :", result["category"])
print("Confidence :", result["confidence"])
print("SLA Hours :", sla)

Category : Refund
Confidence : 5
SLA Hours : 48


In [29]:
#Export CSV of 10 test tickets with model predictions vs
# ground truth labels and compute accuracy
# CREATE TEST DATASET (10 TICKETS)

import pandas as pd

df = pd.DataFrame({

    "ticket": [
        "I was charged twice for my subscription",
        "Unable to login after password reset",
        "Please refund my annual plan",
        "What are your support timings",
        "Legal complaint regarding data misuse",
        "Invoice amount is incorrect",
        "Password reset link not working",
        "Need refund for cancelled order",
        "How can I update my profile",
        "Security breach reported"
    ],

    "ground_truth": [
        "Billing",
        "Tech",
        "Refund",
        "General",
        "Escalate",
        "Billing",
        "Tech",
        "Refund",
        "General",
        "Escalate"
    ]
})

df

,ticket,ground_truth
0,I was charged twice for my subscription,Billing
1,Unable to login after password reset,Tech
2,Please refund my annual plan,Refund
3,What are your support timings,General
4,Legal complaint regarding data misuse,Escalate
5,Invoice amount is incorrect,Billing
6,Password reset link not working,Tech
7,Need refund for cancelled order,Refund
8,How can I update my profile,General
9,Security breach reported,Escalate


In [30]:
#Model Prediction Function (Using task 3, classifier)
# TICKET CLASSIFIER

def classify_ticket(ticket):

    ticket = ticket.lower()

    if "refund" in ticket:
        return "Refund"

    elif "charged" in ticket or "invoice" in ticket:
        return "Billing"

    elif "login" in ticket or "password" in ticket:
        return "Tech"

    elif "legal" in ticket or "security" in ticket:
        return "Escalate"

    else:
        return "General"


In [31]:
# GENERATE MODEL PREDICTIONS

predictions = []

for ticket in df["ticket"]:

    prediction = classify_ticket(ticket)

    predictions.append(prediction)

df["prediction"] = predictions

df

,ticket,ground_truth,prediction
0,I was charged twice for my subscription,Billing,Billing
1,Unable to login after password reset,Tech,Tech
2,Please refund my annual plan,Refund,Refund
3,What are your support timings,General,General
4,Legal complaint regarding data misuse,Escalate,Escalate
5,Invoice amount is incorrect,Billing,Billing
6,Password reset link not working,Tech,Tech
7,Need refund for cancelled order,Refund,Refund
8,How can I update my profile,General,General
9,Security breach reported,Escalate,Escalate


In [32]:
# COMPUTE ACCURACY


from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    df["ground_truth"],
    df["prediction"]
)

print(f"Accuracy: {accuracy:.2f}")

Accuracy: 1.00


In [33]:
# EXPORT RESULTS TO CSV

df.to_csv(
    "ticket_predictions.csv",
    index=False
)

print("CSV exported successfully!")

CSV exported successfully!


In [34]:
# DOWNLOAD CSV FILE

from google.colab import files

files.download("ticket_predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>